In [5]:
import pandas as pd

In [6]:
# Custom functions
from core.normalization_allocation import normalize_flows, normalize_land_flows
from utils.data_manipulations import build_activity_name, add_site_id

In [7]:
# Step 1: aggregate per site_id, archetype, material_category
# site_sum = (
#     df.groupby(["site_id", "archetypes", "material_category"], as_index=False)
#       .agg(site_intensity=("kg_t_ore", "sum"))
# )
#
# # Step 2: compute min / mean / max across sites for each archetype + material_category
# stats = (
#     site_sum.groupby(["archetypes", "material_category"])
#             .agg(
#                 n_sites=("site_id", "nunique"),
#                 min_kg_per_kg=("site_intensity", "min"),
#                 mean_kg_per_kg=("site_intensity", "mean"),
#                 max_kg_per_kg=("site_intensity", "max"),
#             )
#             .reset_index()
#)

In [8]:
# Cleaned data
energy_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\energy_df.xlsx')
material_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\material_df.xlsx')
biosphere_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\biosphere_df.xlsx')
land_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\land_df.xlsx')
carbon_stock_df = pd.read_excel(r'data\MetalliCan\pre_cleaned_data\carbon_stock_df.xlsx')

In [9]:
# Prices and production data
price_df = pd.read_excel(r'data/Prices/Prices_data.xlsx', sheet_name='data')
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [10]:
# Add site_id to dataframes
production_df = add_site_id(production_df)
energy_df = add_site_id(energy_df)
material_df = add_site_id(material_df)
biosphere_df = add_site_id(biosphere_df)
land_df = add_site_id(land_df)
carbon_stock_df = add_site_id(carbon_stock_df)

In [11]:
energy_id_to_remove = [
# less than 50% of NRJ = GHG
'BC-MAIN-599152a0', # Copper Mountain (cu concentrate)
'ON-MAIN-1f126a43', # Macassa (doré)
'ON-MAIN-7f050560', # Red Lake (doré)
'ON-MAIN-7607a50e', # Young Davidson (doré)

# more than 1;5 x
'NU-MAIN-8b0264c9', # Meliadine (doré)
#'QC-MAIN-9de9bb0d', # Kiena (doré)
'ON-MAIN-cb85213a', # Eagle River (doré)
'ON-MAIN-6e9be24e' # Hemlo (Williams)
]

In [12]:
energy_df = energy_df[~energy_df['site_id'].isin(energy_id_to_remove)]

In [13]:
# Replace subflow_type = Other by diesel
energy_df["subflow_type"] = energy_df["subflow_type"].replace("Other", "Diesel")

In [14]:
# Add activitiy_name to production_df
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)

In [15]:
energy_df = energy_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
material_df = material_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
biosphere_df = biosphere_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')
land_df = land_df.merge(production_df[['site_id', 'activity_name','archetypes']], on='site_id', how='left')
carbon_stock_df = carbon_stock_df.merge(production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes']], on='site_id', how='left')

In [16]:
# To avoid double counting, we only keep the individual GHG emissions, and remove the 'NA - GHG' flows from the Canadian national inventory
biosphere_df = biosphere_df[~((biosphere_df['source_id'] == 'https://www.canada.ca/en/environment-climate-change/services/environmental-indicators/greenhouse-gas-emissions/large-facilities.html; https://open.canada.ca/data/en/dataset/a8ba14b7-7f23-462a-bdbb-83b0ef629823') & (biosphere_df['substance_id'] == 'NA - GHG'))]

In [17]:
# Only keep water consumption
biosphere_df = biosphere_df[~biosphere_df['flow_direction'].isin(['Withdrawal', 'Discharged'])]

In [18]:
substance_CO2 = ['124-38-9']
release_pathway = ['Stationary Fuel Combustion', 'On-site Transportation']
CO2_df = biosphere_df[
    biosphere_df['substance_id'].isin(substance_CO2) &
    biosphere_df['release_pathway'].isin(release_pathway)
]

# Data cleaning and integration

In [19]:
energy_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'value_MJ']
material_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'flow_type', 'subflow_type', 'mass_t']
biosphere_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'compartment_name', 'substance_name', 'flow_direction', 'release_pathway', 'unit', 'value']
land_col = ['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'area_m2', 'operation_periods']

In [20]:
energy_df = energy_df[energy_col]
material_df = material_df[material_col]
biosphere_df = biosphere_df[biosphere_col]
CO2_df = CO2_df[biosphere_col]
land_df = land_df[land_col]

## Create a tailings df

In [21]:
# We create a tailings_df to add the quantity of tailings as a negative material flow
tailings_df = production_df[['site_id', 'activity_name', 'mining_processing_type', 'archetypes', 'Stream', 'tailings_t_mass']]
tailings_df['flow_type'] = 'Material use'
tailings_df['subflow_type'] = 'Tailings'
tailings_df['unit'] = 't'
tailings_df['value'] = - tailings_df['tailings_t_mass']  # negative value for output in LCI
tailings_df.drop(columns=['tailings_t_mass'], inplace=True)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\54366009.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['flow_type'] = 'Material use'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\54366009.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = 'Tailings'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\54366009.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead


In [22]:
def get_tailings_subflow_type(row):
    stream = row['Stream']

    if not isinstance(stream, str):
        return 'Tailings|Other'

    if 'Au-Ag doré' in stream:
        return 'Tailings|Gold'
    elif stream == 'Ni-Cu bulk conc.':
        return 'Tailings|Nickel'
    elif 'Cu conc.' or 'Cu and Mo conc.' or 'Cu and Zn conc.' in stream:
        return 'Tailings|Copper'
    elif 'Yellowcake' in stream:
        return 'Tailings|Uranium'
    else:
        return 'Tailings|Other'

In [23]:
tailings_df['subflow_type'] = tailings_df.apply(get_tailings_subflow_type, axis=1)

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\4113187377.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tailings_df['subflow_type'] = tailings_df.apply(get_tailings_subflow_type, axis=1)


## Cleaning

In [24]:
# Maybe need to differentiate unit and value in energy_df and material_df ?
energy_df['unit'] = 'MJ'
material_df['unit'] = 't'
#tailings_df['unit'] = 't'
land_df['unit'] = 'm2'
energy_df.rename(columns={'value_MJ': 'value'}, inplace=True)
material_df.rename(columns={'mass_t': 'value'}, inplace=True)
land_df.rename(columns={'area_m2': 'value'}, inplace=True)

In [25]:
# To know the % inferred vs 'primary'
energy_df['data_source'] = 'MetalliCan'
material_df['data_source'] = 'MetalliCan'
biosphere_df['data_source'] = 'MetalliCan'
CO2_df['data_source'] = 'MetalliCan'
land_df['data_source'] = 'MetalliCan'
carbon_stock_df['data_source'] = 'MetalliCan'

# Before gap filling

In [26]:
energy_df_bgf = energy_df.copy()
material_df_bgf = material_df.copy()
biosphere_df_bgf = biosphere_df.copy()
land_df_bgf = land_df.copy()
carbon_stock_df_bgf = carbon_stock_df.copy()

In [27]:
# Add NPV to land df, and put 'Unspecified NPV' for missing values
land_df_bgf = land_df_bgf.merge(production_df[['site_id', 'npv']], on='site_id', how='left')
land_df_bgf['npv'] = land_df_bgf['npv'].fillna('Unspecified NPV')

## Integrate carbon stock change and water in the relevant dfs

In [28]:
# We integrate the carbon_stock_df in the biosphere_df
carbon_stock_df_bgf = carbon_stock_df_bgf[carbon_stock_df_bgf['pool'] == 'all']
carbon_stock_df_bgf.rename(columns={'value': 'carbon_variation_tC_ha'}, inplace=True)
carbon_stock_df_bgf['carbon_variation_CO2_m2'] = carbon_stock_df_bgf['carbon_variation_tC_ha'] * 44 / 12 / 10000  # convert from C to CO2
carbon_stock_df_bgf.drop(columns=['carbon_variation_tC_ha', 'unit', 'carbon_stock_ecosystems_id', 'pool',  ], inplace=True)
#carbon_stock_df.rename(columns={'carbon_variation_CO2_m2': 'value'}, inplace=True)
carbon_stock_df_bgf['flow_direction'] = 'Emission'
carbon_stock_df_bgf['compartment_name'] = 'Air, soil'
carbon_stock_df_bgf['release_pathway'] = ''
carbon_stock_df_bgf['unit'] = 't/m2'
carbon_stock_df_bgf['substance_name'] = "Carbon stock change"

In [29]:
# Merge with land_df to get the surface area
carbon_stock_df_bgf = carbon_stock_df_bgf.merge(land_df_bgf[['site_id', 'value']], on='site_id', how='left')
#carbon_stock_df.rename(columns={'value': 'area_m2'}, inplace=True)

In [30]:
carbon_stock_df_bgf.rename(columns={'value': 'area_m2'}, inplace=True)

In [31]:
carbon_stock_df_bgf['value'] = carbon_stock_df_bgf['area_m2'] * carbon_stock_df_bgf['carbon_variation_CO2_m2']

In [32]:
carbon_stock_df_bgf = carbon_stock_df_bgf.dropna(subset=['value'])

In [33]:
carbon_stock_df_bgf

,variable,main_id,source_id,facility_group_id,site_id,activity_name,mining_processing_type,archetypes,data_source,carbon_variation_CO2_m2,flow_direction,compartment_name,release_pathway,unit,substance_name,area_m2,value
12,mean_variation,BC-MAIN-23155c25,https://zenodo.org/records/15777016,<NA>,BC-MAIN-23155c25,NaN,NaN,NaN,MetalliCan,0.024933,Emission,"Air, soil",,t/m2,Carbon stock change,1.499690e+06,37392.267311
17,mean_variation,BC-MAIN-3ef4f421,https://zenodo.org/records/15777016,<NA>,BC-MAIN-3ef4f421,NaN,NaN,NaN,MetalliCan,0.017967,Emission,"Air, soil",,t/m2,Carbon stock change,1.396089e+06,25083.063522
18,mean_variation,BC-MAIN-3f490561,https://zenodo.org/records/15777016,<NA>,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator",Cu Porphyry,MetalliCan,0.016500,Emission,"Air, soil",,t/m2,Carbon stock change,7.967835e+06,131469.271520
19,mean_variation,BC-MAIN-4724f4ba,https://zenodo.org/records/15777016,<NA>,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,MetalliCan,0.019433,Emission,"Air, soil",,t/m2,Carbon stock change,4.167369e+05,8098.586319
22,mean_variation,BC-MAIN-599152a0,https://zenodo.org/records/15777016,<NA>,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,MetalliCan,0.018700,Emission,"Air, soil",,t/m2,Carbon stock change,1.323321e+07,247461.034267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,mean_variation,SK-MAIN-9dd2b7f8,https://zenodo.org/records/15777016,<NA>,SK-MAIN-9dd2b7f8,NaN,NaN,NaN,MetalliCan,0.000000,Emission,"Air, soil",,t/m2,Carbon stock change,4.345047e+06,0.000000
252,mean_variation,SK-MAIN-bb89158f,https://zenodo.org/records/15777016,<NA>,SK-MAIN-bb89158f,NaN,NaN,NaN,MetalliCan,0.009167,Emission,"Air, soil",,t/m2,Carbon stock change,1.023565e+07,93826.765900
253,mean_variation,SK-MAIN-d3c471e8,https://zenodo.org/records/15777016,<NA>,SK-MAIN-d3c471e8,NaN,NaN,NaN,MetalliCan,0.008800,Emission,"Air, soil",,t/m2,Carbon stock change,1.973892e+06,17370.253608
260,mean_variation,YT-MAIN-44857446,https://zenodo.org/records/15777016,<NA>,YT-MAIN-44857446,Underground mining and beneficiation at Keno H...,"Underground, concentrator",Ag polymetallic,MetalliCan,0.005133,Emission,"Air, soil",,t/m2,Carbon stock change,5.293594e+06,27173.782646


## Create water df and integrate in biosphere df

In [34]:
water_df_bgf = material_df_bgf[material_df_bgf['flow_type'] == 'Water']

In [35]:
water_df_bgf['substance_name'] = 'Water'
water_df_bgf['compartment_name'] = 'Water'
water_df_bgf['flow_direction'] = 'Consumption'
water_df_bgf['release_pathway'] = ''

In [36]:
water_df_bgf = water_df_bgf[biosphere_col]

In [37]:
biosphere_df_bgf = pd.concat([biosphere_df_bgf, water_df_bgf])

## Normalize flows

In [38]:
energy_conc_econ_bgf = normalize_flows(energy_df_bgf, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [39]:
material_conc_econ_bgf = normalize_flows(material_df_bgf, production_df, price_df=price_df,  mode='concentrate', allocation='economic', value_col='value')

In [40]:
biosphere_conc_econ_bgf = normalize_flows(biosphere_df_bgf, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [41]:
land_conc_econ_bgf = normalize_land_flows(land_df_bgf, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [42]:
carbon_stock_conc_econ_bgf = normalize_land_flows(carbon_stock_df_bgf, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [43]:
land_conc_econ_bgf.loc[
    land_conc_econ_bgf["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [44]:
carbon_stock_conc_econ_bgf.loc[
    carbon_stock_conc_econ_bgf["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [45]:
carbon_stock_conc_econ_bgf = carbon_stock_conc_econ_bgf[carbon_stock_conc_econ_bgf['flow_type'] == 'transformation_from']
carbon_stock_conc_econ_bgf['flow_type'] = 'Carbon stock change'
carbon_stock_conc_econ_bgf['unit'] = 't'

## Exports normalized dataframes

In [46]:
energy_conc_econ_bgf.to_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/energy_df.csv', index=False)
material_conc_econ_bgf.to_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/material_df.csv', index=False)
biosphere_conc_econ_bgf.to_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/biosphere_df.csv', index=False)
land_conc_econ_bgf.to_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/land_df.csv', index=False)
carbon_stock_conc_econ_bgf.to_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/carbon_stock_df.csv', index=False)

# Data-gap filling

In [47]:
from core.data_gap_filling import *

In [48]:
# Initialize the InferenceEngine Class
engine = InferenceEngine(
    site_df=production_df,
    production_df=production_df,
    energy_df=energy_df,
    co2_df=CO2_df,
    land_df=land_df,
    material_df=material_df,
)

## Energy

In [49]:
site_id_to_fill_nrj = production_df[production_df['infer_energy_data'] == 'Yes']['site_id'].tolist()

In [50]:
ef = {
    "diesel": 2681,
    "natural_gas": 2354,
    "lpg": 2753
}

stationary_share_rules = {
    "Open-pit, concentrator": {"diesel": 0.7, "natural_gas": 0.2, "lpg": 0.1},
    "Underground, concentrator": {"diesel": 0.7, "natural_gas": 0.2, "lpg": 0.1},
}

default_shares = {"diesel": 0.3, "natural_gas": 0.6, "lpg": 0.1}

In [51]:
engine.infer_energy_for_sites(
    site_ids=site_id_to_fill_nrj,
    ef_co2_per_unit=ef,
    stationary_share_rules=stationary_share_rules,
    default_shares=default_shares
)

(              site_id                                      activity_name  \
 0    BC-MAIN-857b7b89  Underground mining and beneficiation at Brucejack   
 1    BC-MAIN-857b7b89  Underground mining and beneficiation at Brucejack   
 2    BC-MAIN-857b7b89  Underground mining and beneficiation at Brucejack   
 3    BC-MAIN-857b7b89  Underground mining and beneficiation at Brucejack   
 4    BC-MAIN-857b7b89  Underground mining and beneficiation at Brucejack   
 ..                ...                                                ...   
 225      GRP-147b3123  Underground mining and beneficiation at Timmin...   
 226      GRP-a13779f8  Underground mining and beneficiation at Snow Lake   
 227      GRP-a13779f8  Underground mining and beneficiation at Snow Lake   
 228      GRP-a13779f8  Underground mining and beneficiation at Snow Lake   
 229      GRP-a13779f8  Underground mining and beneficiation at Snow Lake   
 
         mining_processing_type          archetypes flow_type  \
 0    Und

## Electricity

In [52]:
electricity_share_rules = {
    "open-pit": {
        "default": (0.2, 0.4),
    },
    "underground": {
        "default": (0.4, 0.6),
    },
    "mixed" : {
        "default": (0.5, 0.7)
    }
}


In [53]:
combined_energy_df, inferred_electricity_df = engine.infer_electricity_for_sites(
    site_ids=site_id_to_fill_nrj,
    electricity_share_rules=electricity_share_rules,
    overwrite=False,
)

⚠️ No electricity share rule for None
⚠️ No electricity share rule for None
⚠️ No electricity share rule for None
⚠️ No electricity share rule for None
⚠️ No electricity share rule for None
⚠️ No electricity share rule for None
⚠️ No electricity share rule for None


In [54]:
inferred_electricity_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,parameter_distribution
0,QC-MAIN-089f3c60,Open-pit mining at Bloom Lake,Open-pit,Magnetite concentrator,Energy,Electricity consumption|Grid electricity,6.386060e+08,MJ,Inference | electricity share of total energy,(0.30000000000000004 / (1 - 0.3000000000000000...,"Uniform(0.2, 0.4)"
1,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,Energy,Electricity consumption|Grid electricity,4.263267e+08,MJ,Inference | electricity share of total energy,(0.30000000000000004 / (1 - 0.3000000000000000...,"Uniform(0.2, 0.4)"
2,ON-MAIN-cb85213a,Underground mining and beneficiation at Eagle ...,"Underground, concentrator",Au–Ag free-milling,Energy,Electricity consumption|Grid electricity,1.569130e+08,MJ,Inference | electricity share of total energy,(0.5 / (1 - 0.5)) * fuel_energy_MJ,"Uniform(0.4, 0.6)"
3,ON-MAIN-6e9be24e,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Au–Ag free-milling,Energy,Electricity consumption|Grid electricity,2.726562e+08,MJ,Inference | electricity share of total energy,(0.6 / (1 - 0.6)) * fuel_energy_MJ,"Uniform(0.5, 0.7)"
4,BC-MAIN-bf503b6b,Open-pit mining and beneficiation at Highland ...,"Open-pit, concentrator",Cu polymetallic,Energy,Electricity consumption|Grid electricity,1.064712e+09,MJ,Inference | electricity share of total energy,(0.30000000000000004 / (1 - 0.3000000000000000...,"Uniform(0.2, 0.4)"
5,ON-MAIN-1f126a43,Underground mining and beneficiation at Macassa,"Underground, concentrator",Au–Ag free-milling,Energy,Electricity consumption|Grid electricity,2.803699e+08,MJ,Inference | electricity share of total energy,(0.5 / (1 - 0.5)) * fuel_energy_MJ,"Uniform(0.4, 0.6)"
6,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Energy,Electricity consumption|Grid electricity,4.440703e+08,MJ,Inference | electricity share of total energy,(0.30000000000000004 / (1 - 0.3000000000000000...,"Uniform(0.2, 0.4)"
7,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator",Cu Porphyry,Energy,Electricity consumption|Grid electricity,2.915982e+08,MJ,Inference | electricity share of total energy,(0.30000000000000004 / (1 - 0.3000000000000000...,"Uniform(0.2, 0.4)"
8,QC-MAIN-a555d9b9,Underground mining and beneficiation at Niobec,"Underground, concentrator",Nb mine,Energy,Electricity consumption|Grid electricity,1.415579e+08,MJ,Inference | electricity share of total energy,(0.5 / (1 - 0.5)) * fuel_energy_MJ,"Uniform(0.4, 0.6)"
9,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Ni–Cu sulfide,Energy,Electricity consumption|Grid electricity,1.317082e+09,MJ,Inference | electricity share of total energy,(0.6 / (1 - 0.6)) * fuel_energy_MJ,"Uniform(0.5, 0.7)"


## Material

In [55]:
site_id_to_fill_material = production_df[production_df['infer_material_data'] == 'Yes']['site_id'].tolist()

In [56]:
material_archetype_rules_df = pd.read_excel(r'data/SI/SI_3_data_gap_filling.xlsx', sheet_name='DATA')

In [57]:
engine.init_material_inference(material_archetype_rules_df)

In [58]:
combined_material_df, inferred_material_df = engine.infer_material_for_sites(
    site_ids=site_id_to_fill_material,
    overwrite=False,
)

⚠️ No material rules for archetype 'Magnetite concentrator'
⚠️ No ore_processed_t for site QC-MAIN-de3d8b7b
⚠️ No material rules for archetype 'Zn refinery'
⚠️ No material rules for archetype 'Fe concentrator + pellet plant'
⚠️ No ore_processed_t for site ON-MAIN-63b394c3
⚠️ No material rules for archetype 'Cu-Ni smelter and refinery'
⚠️ No ore_processed_t for site QC-MAIN-30c1828c
⚠️ No material rules for archetype 'Cu smelter'
⚠️ No ore_processed_t for site NL-MAIN-d9036091
⚠️ No material rules for archetype 'Ni hydrometallurgical refinery'
⚠️ No material rules for archetype 'Direct Shipping Ore'
⚠️ No material rules for archetype 'Nb mine'
⚠️ No ore_processed_t for site QC-MAIN-649d2873
⚠️ No material rules for archetype 'Nb smelter'
⚠️ No material rules for archetype 'Fe concentrator'
⚠️ No ore_processed_t for site ON-MAIN-40ce0593
⚠️ No material rules for archetype 'Ni-Cu smelter'
⚠️ No ore_processed_t for site AB-MAIN-d3a4aba9
⚠️ No material rules for archetype 'Co hydrometallurg

## Cement

In [59]:
site_id_to_fill_material = production_df[production_df['infer_cement_data'] == 'Yes']['site_id'].tolist()

In [64]:
cement_params = {
    "underground": {
        "cement_factor": (1.6, 3.5), # Eleonore and Musselwhite
        #"backfill_share": (0.3, 0.9),  # optional, future
    },
    "open_pit": (1.5, 1.5), # Porcupine Complex
}


In [65]:
combined_cement_df, inferred_cement_df = engine.infer_cement_for_sites(
    site_ids=site_id_to_fill_material,
    cement_params=cement_params,
)

## Explosives

In [66]:
site_id_to_fill_explosives = production_df[production_df['infer_explosives_data'] == 'Yes']['site_id'].tolist()

In [67]:
explosive_archetype_rules_df = pd.read_excel(r'data/SI/SI_3_data_gap_filling.xlsx', sheet_name='DATA_explosives')

In [68]:
engine.init_material_inference(explosive_archetype_rules_df)

In [69]:
combined_explosives_df, inferred_explosives_df = engine.infer_material_for_sites(
    site_ids=site_id_to_fill_explosives,
    overwrite=False,
)

⚠️ No material rules for archetype 'Magnetite concentrator'
⚠️ No material rules for archetype 'Fe concentrator + pellet plant'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Direct Shipping Ore'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Nb mine'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Fe concentrator'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'Ni–Cu sulfide'
⚠️ No material rules for archetype 'High-grade U mine + mill'
⚠️ No material rules for archetype 'High-gr

In [70]:
# explosives_params = {
#     "open_pit": {
#         "strip_ratio": (1.5, 6.0),       # t waste / t ore
#         "explosive_factor": (0.25, 0.8), # kg explosives / t material
#     },
#     "underground": {
#         "explosive_factor": (0.1, 0.3),  # kg explosives / t ore
#     }
# }

In [71]:
# combined_explosives_df, inferred_explosives_df = engine.infer_explosives_for_sites(
#     site_ids=site_id_to_fill_explosives,
#     explosive_params=explosives_params,
# )

## Land

In [72]:
site_id_to_fill_land = production_df[production_df['infer_land_data'] == 'Yes']['site_id'].tolist()

In [73]:
underground_land_factors = {
    "Au–Ag free-milling": {
        "factor": (1.5),   # m² / t ore
    },
    "Ni–Cu sulfide": {
        "factor": (3.18),
    },
    "Cu Porphyry": {
        "factor": (1.61),
    },
    "Cu polymetallic": {
        "factor": (3.13),
    },
    "Au–Ag polymetallic": {
        "factor": (1.46),
    }
}

In [74]:
combined_land_df, inferred_land_df = engine.infer_land_for_sites(
    site_ids=site_id_to_fill_land,
    formula_open_pit="3.84e3 * ore_processed_t **0.51",
    underground_land_factors=underground_land_factors,
    formula_other="Uniform(1e4, 1e5)",
    overwrite=False
)

In [75]:
inferred_land_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,parameter_distribution
0,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Au–Ag free-milling,Land,Land area,2.490000e+05,m2,Inference | land | underground | Au–Ag free-mi...,ore_processed_t * 1.5,None
1,QC-MAIN-30c1828c,Smelting at Horne,Smelter,Cu smelter,Land,Land area,NaN,m2,Inference | land | other facilities,"ore_processed_t * Uniform(1e4, 1e5)","Uniform(1e4, 1e5)"
2,ON-MAIN-687b8c8d,Underground mining and beneficiation at Island,"Underground, concentrator",Au–Ag free-milling,Land,Land area,6.585120e+05,m2,Inference | land | underground | Au–Ag free-mi...,ore_processed_t * 1.5,None
3,NL-MAIN-d9036091,Refining at Long Harbour,Refinery,Ni hydrometallurgical refinery,Land,Land area,NaN,m2,Inference | land | other facilities,"ore_processed_t * Uniform(1e4, 1e5)","Uniform(1e4, 1e5)"
4,NU-MAIN-5154702a,Open-pit mining at Mary River,Open-pit,Direct Shipping Ore,Land,Land area,1.061469e+07,m2,Inference | land | open-pit regression,3.84e3 * ore_processed_t **0.51,None
5,NU-MAIN-8b0264c9,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Au–Ag free-milling,Land,Land area,6.146113e+06,m2,Inference | land | open-pit regression,3.84e3 * ore_processed_t **0.51,None
6,ON-MAIN-fcb287a4,Underground mining at Nickel Rim South,Underground,Ni–Cu sulfide,Land,Land area,1.018109e+06,m2,Inference | land | underground | Ni–Cu sulfide,ore_processed_t * 3.18,None
7,QC-MAIN-649d2873,Converting at Niobec Converter,Convertor,Nb smelter,Land,Land area,NaN,m2,Inference | land | other facilities,"ore_processed_t * Uniform(1e4, 1e5)","Uniform(1e4, 1e5)"
8,QC-MAIN-5ce331b8,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Ni–Cu sulfide,Land,Land area,1.175053e+07,m2,Inference | land | open-pit regression,3.84e3 * ore_processed_t **0.51,None
9,ON-MAIN-40ce0593,Smelting at Sudbury,"Smelter, plant",Ni-Cu smelter,Land,Land area,NaN,m2,Inference | land | other facilities,"ore_processed_t * Uniform(1e4, 1e5)","Uniform(1e4, 1e5)"


In [76]:
# Add NPV to land df, and put 'Unspecified NPV' for missing values
combined_land_df = combined_land_df.merge(production_df[['site_id', 'npv']], on='site_id', how='left')

In [77]:
combined_land_df['npv'] = combined_land_df['npv'].fillna('Unspecified NPV')

In [78]:
combined_land_df

,site_id,activity_name,mining_processing_type,archetypes,value,operation_periods,unit,data_source,flow_type,subflow_type,value_formula,parameter_distribution,npv
0,BC-MAIN-23155c25,NaN,Underground,NaN,1.499690e+06,1966–1985; 2002–2015; 2019–open,m2,MetalliCan,NaN,NaN,NaN,NaN,Unspecified NPV
1,BC-MAIN-3ef4f421,NaN,NaN,NaN,1.396089e+06,NaN,m2,MetalliCan,NaN,NaN,NaN,NaN,Unspecified NPV
2,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator",Cu Porphyry,7.967835e+06,NaN,m2,MetalliCan,NaN,NaN,NaN,NaN,Cold Evergreen Needleleaf Forest
3,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,4.167369e+05,NaN,m2,MetalliCan,NaN,NaN,NaN,NaN,Cold Evergreen Needleleaf Forest
4,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,1.323321e+07,1884–1958; 2011–open,m2,MetalliCan,NaN,NaN,NaN,NaN,Cool Evergreen Needleleaf Forest
...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Au–Ag free-milling,3.520065e+05,NaN,m2,Inference | land | underground | Au–Ag free-mi...,Land,Land area,ore_processed_t * 1.5,None,Cool Mixed Forest
142,AB-MAIN-d3a4aba9,Refining at The Cobalt Refinery Company Inc.,Refinery,Co hydrometallurgical refinery,NaN,NaN,m2,Inference | land | other facilities,Land,Land area,"ore_processed_t * Uniform(1e4, 1e5)","Uniform(1e4, 1e5)",Cold Evergreen Needleleaf Forest
143,MB-MAIN-e0a6250e,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Ni–Cu sulfide,2.982580e+06,NaN,m2,Inference | land | open-pit regression,Land,Land area,3.84e3 * ore_processed_t **0.51,None,Cold Evergreen Needleleaf Forest
144,BC-MAIN-3bb6b7cd,Refining at Trail,Refinery,Zn refinery,NaN,NaN,m2,Inference | land | other facilities,Land,Land area,"ore_processed_t * Uniform(1e4, 1e5)","Uniform(1e4, 1e5)",Cool Evergreen Needleleaf Forest


# Integrate carbon stock change and water in the relevant dfs

## Multiply land area with carbon stock change and integrate in land_df

In [79]:
carbon_stock_df

,carbon_stock_ecosystems_id,pool,variable,unit,value,main_id,source_id,facility_group_id,site_id,activity_name,mining_processing_type,archetypes,data_source
0,carbon_stock-35d0dc71-1,agbc,area_ha,ha,49.315624,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan
1,carbon_stock-35d0dc71-2,agbc,mean_act,tC/ha,10.000000,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan
2,carbon_stock-35d0dc71-3,agbc,mean_prim,tC/ha,36.000000,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan
3,carbon_stock-35d0dc71-4,bgbc,area_ha,ha,49.315624,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan
4,carbon_stock-35d0dc71-5,bgbc,mean_act,tC/ha,3.000000,AB-MAIN-35d0dc71,https://zenodo.org/records/15777016,<NA>,AB-MAIN-35d0dc71,NaN,NaN,NaN,MetalliCan
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3640,NaN,all,mean_variation,tC/ha,24.000000,YT-MAIN-752fbcae,https://zenodo.org/records/15777016,<NA>,YT-MAIN-752fbcae,NaN,NaN,NaN,MetalliCan
3641,NaN,all,mean_variation,tC/ha,2.000000,YT-MAIN-956d050a,https://zenodo.org/records/15777016,<NA>,YT-MAIN-956d050a,NaN,NaN,NaN,MetalliCan
3642,NaN,all,mean_variation,tC/ha,2.000000,YT-MAIN-c3105f43,https://zenodo.org/records/15777016,<NA>,YT-MAIN-c3105f43,NaN,NaN,NaN,MetalliCan
3643,NaN,all,mean_variation,tC/ha,0.000000,YT-MAIN-ca488f19,https://zenodo.org/records/15777016,<NA>,YT-MAIN-ca488f19,NaN,NaN,NaN,MetalliCan


In [80]:
# We integrate the carbon_stock_df in the biosphere_df
carbon_stock_df = carbon_stock_df[carbon_stock_df['pool'] == 'all']
carbon_stock_df.rename(columns={'value': 'carbon_variation_tC_ha'}, inplace=True)
carbon_stock_df['carbon_variation_CO2_m2'] = carbon_stock_df['carbon_variation_tC_ha'] * 44 / 12 / 10000  # convert from C to CO2
carbon_stock_df.drop(columns=['carbon_variation_tC_ha', 'unit', 'carbon_stock_ecosystems_id', 'pool',  ], inplace=True)
#carbon_stock_df.rename(columns={'carbon_variation_CO2_m2': 'value'}, inplace=True)
carbon_stock_df['flow_direction'] = 'Emission'
carbon_stock_df['compartment_name'] = 'Air, soil'
carbon_stock_df['release_pathway'] = ''
carbon_stock_df['unit'] = 't/m2'
carbon_stock_df['substance_name'] = "Carbon stock change"

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\1528102575.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  carbon_stock_df.rename(columns={'value': 'carbon_variation_tC_ha'}, inplace=True)
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\1528102575.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  carbon_stock_df['carbon_variation_CO2_m2'] = carbon_stock_df['carbon_variation_tC_ha'] * 44 / 12 / 10000  # convert from C to CO2
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\1528102575.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [81]:
# Merge with land_df to get the surface area
carbon_stock_df = carbon_stock_df.merge(combined_land_df[['site_id', 'value']], on='site_id', how='left')
#carbon_stock_df.rename(columns={'value': 'area_m2'}, inplace=True)

In [82]:
carbon_stock_df.rename(columns={'value': 'area_m2'}, inplace=True)

In [83]:
carbon_stock_df['value'] = carbon_stock_df['area_m2'] * carbon_stock_df['carbon_variation_CO2_m2']

In [84]:
carbon_stock_df = carbon_stock_df.dropna(subset=['value'])

In [85]:
carbon_stock_df

,variable,main_id,source_id,facility_group_id,site_id,activity_name,mining_processing_type,archetypes,data_source,carbon_variation_CO2_m2,flow_direction,compartment_name,release_pathway,unit,substance_name,area_m2,value
12,mean_variation,BC-MAIN-23155c25,https://zenodo.org/records/15777016,<NA>,BC-MAIN-23155c25,NaN,NaN,NaN,MetalliCan,0.024933,Emission,"Air, soil",,t/m2,Carbon stock change,1.499690e+06,37392.267311
17,mean_variation,BC-MAIN-3ef4f421,https://zenodo.org/records/15777016,<NA>,BC-MAIN-3ef4f421,NaN,NaN,NaN,MetalliCan,0.017967,Emission,"Air, soil",,t/m2,Carbon stock change,1.396089e+06,25083.063522
18,mean_variation,BC-MAIN-3f490561,https://zenodo.org/records/15777016,<NA>,BC-MAIN-3f490561,Open-pit mining and beneficiation at Mount Polley,"Open-pit, concentrator",Cu Porphyry,MetalliCan,0.016500,Emission,"Air, soil",,t/m2,Carbon stock change,7.967835e+06,131469.271520
19,mean_variation,BC-MAIN-4724f4ba,https://zenodo.org/records/15777016,<NA>,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,MetalliCan,0.019433,Emission,"Air, soil",,t/m2,Carbon stock change,4.167369e+05,8098.586319
22,mean_variation,BC-MAIN-599152a0,https://zenodo.org/records/15777016,<NA>,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,MetalliCan,0.018700,Emission,"Air, soil",,t/m2,Carbon stock change,1.323321e+07,247461.034267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,mean_variation,SK-MAIN-9dd2b7f8,https://zenodo.org/records/15777016,<NA>,SK-MAIN-9dd2b7f8,NaN,NaN,NaN,MetalliCan,0.000000,Emission,"Air, soil",,t/m2,Carbon stock change,4.345047e+06,0.000000
258,mean_variation,SK-MAIN-bb89158f,https://zenodo.org/records/15777016,<NA>,SK-MAIN-bb89158f,NaN,NaN,NaN,MetalliCan,0.009167,Emission,"Air, soil",,t/m2,Carbon stock change,1.023565e+07,93826.765900
259,mean_variation,SK-MAIN-d3c471e8,https://zenodo.org/records/15777016,<NA>,SK-MAIN-d3c471e8,NaN,NaN,NaN,MetalliCan,0.008800,Emission,"Air, soil",,t/m2,Carbon stock change,1.973892e+06,17370.253608
266,mean_variation,YT-MAIN-44857446,https://zenodo.org/records/15777016,<NA>,YT-MAIN-44857446,Underground mining and beneficiation at Keno H...,"Underground, concentrator",Ag polymetallic,MetalliCan,0.005133,Emission,"Air, soil",,t/m2,Carbon stock change,5.293594e+06,27173.782646


## Create water df and integrate in biosphere df

In [86]:
water_df = combined_material_df[combined_material_df['flow_type'] == 'Water']

In [87]:
water_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,parameter_distribution
82,BC-MAIN-857b7b89,Underground mining and beneficiation at Brucejack,"Underground, concentrator",Au–Ag free-milling,Water,Water,1.035403e+05,kg,Archetype inference | MetalliCan,ore_processed_t * 0.6237369273366383,<NA>
92,QC-MAIN-e7e6a960,Open-pit mining and beneficiation at Canadian ...,"Open-pit, concentrator",Au–Ag free-milling,Water,Water,1.081117e+07,kg,Archetype inference | MetalliCan,ore_processed_t * 0.6237369273366383,<NA>
98,ON-MAIN-f080c409,Underground mining at Copper Cliff Complex (mine),Underground,Ni–Cu sulfide,Water,Water,6.907546e+05,m3,Archetype inference | MetalliCan,ore_processed_t * 0.9234686820974333,<NA>
105,BC-MAIN-599152a0,Open-pit mining and beneficiation at Copper Mo...,"Open-pit, concentrator",Cu Porphyry,Water,Water,6.336982e+06,m3,Archetype inference | MetalliCan,ore_processed_t * 0.9234686820974333,<NA>
111,ON-MAIN-52224e1e,Underground mining at Creighton,Underground,Ni–Cu sulfide,Water,Water,3.998619e+05,m3,Archetype inference | MetalliCan,ore_processed_t * 0.9234686820974333,<NA>
121,ON-MAIN-aeafbb59,Open-pit mining and beneficiation at Detour Lake,"Open-pit, concentrator",Au–Ag free-milling,Water,Water,1.586466e+07,kg,Archetype inference | MetalliCan,ore_processed_t * 0.6237369273366383,<NA>
131,BC-MAIN-4724f4ba,Open-pit mining at Elk,Open-pit,Au–Ag free-milling,Water,Water,2.073613e+04,kg,Archetype inference | MetalliCan,ore_processed_t * 0.6237369273366383,<NA>
141,ON-MAIN-4e0734b5,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Au–Ag free-milling,Water,Water,2.850478e+05,kg,Archetype inference | MetalliCan,ore_processed_t * 0.6237369273366383,<NA>
147,ON-MAIN-206041d1,Underground mining at Fraser,Underground,Ni–Cu sulfide,Water,Water,5.107087e+05,m3,Archetype inference | MetalliCan,ore_processed_t * 0.9234686820974333,<NA>
153,ON-MAIN-48fe2205,Underground mining at Garson,Underground,Ni–Cu sulfide,Water,Water,5.688567e+05,m3,Archetype inference | MetalliCan,ore_processed_t * 0.9234686820974333,<NA>


In [88]:
water_df['substance_name'] = 'Water'
water_df['compartment_name'] = 'Water'
water_df['flow_direction'] = 'Consumption'
water_df['release_pathway'] = ''

C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\1603994030.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  water_df['substance_name'] = 'Water'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\1603994030.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  water_df['compartment_name'] = 'Water'
C:\Users\mp_ma\AppData\Local\Temp\ipykernel_964\1603994030.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



In [89]:
water_df = water_df[biosphere_col]

In [90]:
biosphere_df = pd.concat([biosphere_df, water_df])

# Normalize flows

In [91]:
combined_explosives_df = combined_explosives_df[combined_explosives_df['flow_type'] != 'Water']

In [92]:
combined_explosives_df['flow_type'].unique()

array(['Material use', 'Materials', 'Material'], dtype=object)

In [93]:
combined_explosives_df

,site_id,activity_name,mining_processing_type,archetypes,flow_type,subflow_type,value,unit,data_source,value_formula,parameter_distribution
0,QC-MAIN-b86f7d07,Open-pit and underground mining and beneficiat...,"Open-pit, underground, concentrator",Au–Ag free-milling,Material use,Surface/underground emulsion & ANFO,2.968124e+03,t,MetalliCan,NaN,NaN
1,ON-MAIN-aeafbb59,Open-pit mining and beneficiation at Detour Lake,"Open-pit, concentrator",Au–Ag free-milling,Material use,Explosives,1.626850e+04,t,MetalliCan,NaN,NaN
2,ON-MAIN-cb85213a,Underground mining and beneficiation at Eagle ...,"Underground, concentrator",Au–Ag free-milling,Material use,Explosives,1.211000e+03,t,MetalliCan,NaN,NaN
3,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Au–Ag free-milling,Material use,Cement,2.737400e+04,t,MetalliCan,NaN,NaN
4,QC-MAIN-6dc537e6,Underground mining and beneficiation at Éléonore,"Underground, concentrator",Au–Ag free-milling,Material use,Grinding media,3.039900e+03,t,MetalliCan,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
452,ON-MAIN-7f050560,Underground mining and beneficiation at Red Lake,"Underground, concentrator",Au–Ag refractory,Materials,Explosives,4.036800e+05,kg,Archetype inference | MetalliCan,ore_processed_t * 0.48,<NA>
453,ON-MAIN-f4fc3276,Underground mining and beneficiation at Sugar ...,"Underground, concentrator",Au–Ag free-milling,Materials,Explosives,1.126421e+05,kg,Archetype inference | MetalliCan,ore_processed_t * 0.48,<NA>
454,GRP-0d911886,Open-pit and underground mining at Porcupine c...,"Open-pit, underground",Au–Ag free-milling,Materials,Explosives,1.397280e+06,kg,Archetype inference | MetalliCan,ore_processed_t * 0.48,<NA>
455,GRP-14bfbb82,Underground mining and beneficiation at Seabee...,"Underground, concentrator",Au–Ag free-milling,Materials,Explosives,5.856000e+04,kg,Archetype inference | MetalliCan,ore_processed_t * 0.48,<NA>


In [94]:
combined_material_df = pd.concat([combined_explosives_df, tailings_df], ignore_index=True)

In [95]:
combined_material_df['flow_type'].unique()

array(['Material use', 'Materials', 'Material'], dtype=object)

In [96]:
from core.normalization_allocation import normalize_flows, normalize_land_flows

### Per ore processed

In [97]:
energy_ore = normalize_flows(energy_df, production_df, mode='ore', value_col='value')
material_ore = normalize_flows(material_df, production_df, mode='ore', value_col='value')
biosphere_ore = normalize_flows(biosphere_df, production_df, mode='ore', value_col='value')

### Per concentrate stream

In [98]:
energy_conc_econ = normalize_flows(combined_energy_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [99]:
material_conc_econ = normalize_flows(combined_material_df, production_df, price_df=price_df,  mode='concentrate', allocation='economic', value_col='value')

In [100]:
biosphere_conc_econ = normalize_flows(biosphere_df, production_df, price_df=price_df, mode='concentrate', allocation='economic', value_col='value')

In [101]:
land_conc_econ = normalize_land_flows(combined_land_df, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [102]:
carbon_stock_conc_econ = normalize_land_flows(carbon_stock_df, production_df, price_df=price_df, mode='concentrate', allocation='economic')

In [103]:
land_conc_econ.loc[
    land_conc_econ["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [104]:
carbon_stock_conc_econ.loc[
    carbon_stock_conc_econ["functional_unit"].str.lower().eq("au concentrate"),
    "functional_unit"
] = "Doré"

In [105]:
carbon_stock_conc_econ = carbon_stock_conc_econ[carbon_stock_conc_econ['flow_type'] == 'transformation_from']
carbon_stock_conc_econ['flow_type'] = 'Carbon stock change'
carbon_stock_conc_econ['unit'] = 't'

# Exports normalized dataframes

In [106]:
energy_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/energy_df.csv', index=False)
material_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/material_df.csv', index=False)
biosphere_ore.to_csv(r'data/MetalliCan/data_for_lci_initialization/ore/biosphere_df.csv', index=False)

In [107]:
energy_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv', index=False)
material_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv', index=False)
biosphere_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv', index=False)
land_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/land_df.csv', index=False)
carbon_stock_conc_econ.to_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/carbon_stock_df.csv', index=False)